## Configuration and Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import AutoTokenizer

# --- CONFIGURATION ---
MODEL_ID = "meta-llama/Meta-Llama-3.1-8B"
DATASET_PATH = "sahil2801/CodeAlpaca-20k"
DATASET_SPLIT = "train"
SYSTEM_PROMPT = "You are an expert programming assistant. Write clean, efficient, and well-documented code."
CODE_SIGNALS = [
    "def ",
    "import ",
    "class ",
    "return ",
    "print(",
    "{",
    "const ",
    "var ",
    "<html>",
    "SELECT ",
    "FROM ",
    "#include",
    "public ",
    "static ",
    "void ",
    "if ",
    "for ",
    "while ",
    "let ",
    "arr =",
    "lst =",
    " = ",
]


## EDA

In [ ]:
def run_codealpaca_eda():
    print("1. Loading Dataset and Tokenizer...")
    dataset = load_dataset(DATASET_PATH, split=DATASET_SPLIT)
    train_df = dataset.to_pandas()

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

    print("\n2. Analyzing Column Artifacts...")
    train_df["has_code_signal"] = train_df["output"].apply(
        lambda x: any(kw in str(x) for kw in CODE_SIGNALS)
    )

    low_signal_df = train_df[train_df["has_code_signal"] == False]
    print(f"Total Rows: {len(train_df)}")
    print(f"Rows with Code Signal: {len(train_df) - len(low_signal_df)}")
    print(
        f"Rows lacking Code Signal: {len(low_signal_df)} ({(len(low_signal_df) / len(train_df)) * 100:.2f}%)"
    )

    print("\n3. Completion (Output) Length Analysis...")

    def get_completion_length(text):
        tokens = tokenizer.encode(str(text), add_special_tokens=False)
        return len(tokens)

    train_df["output_token_len"] = train_df["output"].apply(get_completion_length)
    print("Output Token Length Percentiles:")
    print(train_df["output_token_len"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

    print("\n4. Calculating Total Sequence Length (Context Window Usage)...")

    def calculate_total_lengths(row):
        user_content = str(row.get("instruction", "")).strip()
        raw_input = str(row.get("input", "")).strip()

        if raw_input and raw_input.lower() != "< noinput >":
            user_content += f"\n\nContext/Input:\n{raw_input}"

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": str(row.get("output", "")).strip()},
        ]

        formatted_string = tokenizer.apply_chat_template(messages, tokenize=False)
        total_tokens = len(tokenizer.encode(formatted_string))

        prompt_only = messages[:2]
        formatted_prompt = tokenizer.apply_chat_template(prompt_only, tokenize=False)
        prompt_tokens = len(tokenizer.encode(formatted_prompt))

        return pd.Series([prompt_tokens, (total_tokens - prompt_tokens), total_tokens])

    train_df[["prompt_tokens", "completion_tokens", "total_tokens"]] = train_df.apply(
        calculate_total_lengths, axis=1
    )

    print("\nFull Sequence Percentiles (Prompt + Output):")
    print(train_df[["total_tokens"]].describe(percentiles=[0.50, 0.90, 0.95, 0.99]))

    print("\n5. Visualizing distributions...")
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    sns.histplot(
        train_df["completion_tokens"], bins=50, ax=axes[0], color="orange"
    ).set_title("Output (Completion) Length")
    sns.histplot(
        train_df["total_tokens"], bins=50, ax=axes[1], color="green"
    ).set_title("Total Sequence Length")
    plt.show()


## Execution

In [ ]:
if __name__ == "__main__":
    run_codealpaca_eda()
